In [72]:
import torch

cuda_available = torch.cuda.is_available()

if cuda_available:
    print(f"CUDA is available.\n{torch.cuda.device_count()}, {torch.cuda.get_device_name(torch.cuda.current_device())}")
else:
    print("CUDA is not available.")

CUDA is available.
1, Tesla T4


In [73]:
#hide
# ! [ -e /content ] && pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

In [74]:
#hide
from fastbook import *
from IPython.display import display,HTML

# NLP Deep Dive: RNNs

In <<chapter_intro>> we saw that deep learning can be used to get great results with natural language datasets. Our example relied on using a pretrained language model and fine-tuning it to classify reviews. That example highlighted a difference between transfer learning in NLP and computer vision: in general in NLP the pretrained model is trained on a different task.

What we call a language model is a model that has been trained to guess what the next word in a text is (having read the ones before). This kind of task is called *self-supervised learning*: we do not need to give labels to our model, just feed it lots and lots of texts. It has a process to automatically get labels from the data, and this task isn't trivial: to properly guess the next word in a sentence, the model will have to develop an understanding of the English (or other) language. Self-supervised learning can also be used in other domains; for instance, see ["Self-Supervised Learning and Computer Vision"](https://www.fast.ai/2020/01/13/self_supervised/) for an introduction to vision applications. Self-supervised learning is not usually used for the model that is trained directly, but instead is used for pretraining a model used for transfer learning.

在《章节简介》中我们看到，深度学习可用于处理自然语言数据集并取得出色的成果。我们的示例依赖于使用一个预训练的语言模型，并对其进行微调以对评论进行分类。该示例凸显了自然语言处理（NLP）中的迁移学习与计算机视觉中的迁移学习之间的一个区别：一般来说，在自然语言处理中，预训练模型是在不同的任务上进行训练的。

我们所说的语言模型是一种经过训练的模型，它能够（在读取了前面的单词之后）猜测文本中的下一个单词是什么。这种任务被称为 “自监督学习”：我们无需给模型提供标签，只需向它输入大量大量的文本即可。它有一个从数据中自动获取标签的过程，而且这项任务并非轻而易举：为了准确地猜测句子中的下一个单词，该模型将必须培养对英语（或其他语言）的理解能力。自监督学习也可以应用于其他领域；例如，可查看[《自监督学习与计算机视觉》](https://www.fast.ai/2020/01/13/self_supervised/)一文，以了解关于视觉应用方面的介绍。自监督学习通常并不用于直接训练的模型，而是用于对一个用于迁移学习的模型进行预训练。

> jargon: Self-supervised learning: Training a model using labels that are embedded in the independent variable, rather than requiring external labels. For instance, training a model to predict the next word in a text.

> 术语：自监督学习：利用嵌入在自变量中的标签来训练模型，而无需依赖外部标签。例如，训练一个模型来预测文本中的下一个单词。

The language model we used in <<chapter_intro>> to classify IMDb reviews was pretrained on Wikipedia. We got great results by directly fine-tuning this language model to a movie review classifier, but with one extra step, we can do even better. The Wikipedia English is slightly different from the IMDb English, so instead of jumping directly to the classifier, we could fine-tune our pretrained language model to the IMDb corpus and then use *that* as the base for our classifier.

Even if our language model knows the basics of the language we are using in the task (e.g., our pretrained model is in English), it helps to get used to the style of the corpus we are targeting. It may be more informal language, or more technical, with new words to learn or different ways of composing sentences. In the case of the IMDb dataset, there will be lots of names of movie directors and actors, and often a less formal style of language than that seen in Wikipedia.

We already saw that with fastai, we can download a pretrained English language model and use it to get state-of-the-art results for NLP classification. (We expect pretrained models in many more languages to be available soon—they might well be available by the time you are reading this book, in fact.) So, why are we learning how to train a language model in detail?

One reason, of course, is that it is helpful to understand the foundations of the models that you are using. But there is another very practical reason, which is that you get even better results if you fine-tune the (sequence-based) language model prior to fine-tuning the classification model. For instance, for the IMDb sentiment analysis task, the dataset includes 50,000 additional movie reviews that do not have any positive or negative labels attached. Since there are 25,000 labeled reviews in the training set and 25,000 in the validation set, that makes 100,000 movie reviews altogether. We can use all of these reviews to fine-tune the pretrained language model, which was trained only on Wikipedia articles; this will result in a language model that is particularly good at predicting the next word of a movie review.

This is known as the Universal Language Model Fine-tuning (ULMFit) approach. The [paper](https://arxiv.org/abs/1801.06146) showed that this extra stage of fine-tuning of the language model, prior to transfer learning to a classification task, resulted in significantly better predictions. Using this approach, we have three stages for transfer learning in NLP, as summarized in <<ulmfit_process>>.

我们在《章节简介》中用来对互联网电影数据库（IMDb）影评进行分类的语言模型，是在维基百科的语料上进行预训练的。我们直接对这个语言模型进行微调，将它用作影评分类器，取得了很不错的成果。不过，要是再多做一个步骤，我们还能有更出色的表现。维基百科所使用的英语和IMDb影评里的英语有点不一样，所以我们可以先把预训练好的语言模型在IMDb语料库上微调一下，然后再用微调后的模型作为我们分类器的基础，而不是直接就开始构建分类器。

就算我们的语言模型掌握了在这项任务中所使用语言的基础知识（比如，我们的预训练模型是基于英语的），让它适应我们目标语料库的风格也是很有帮助的。目标语料库的语言可能更口语化、不那么正式，或者专业性更强，会有新的单词要学，也可能有不同的造句方式。就拿IMDb数据集来说吧，里面会有很多电影导演和演员的名字，而且语言风格往往比维基百科上的更随意、没那么正式。

我们已经知道，借助fastai工具，我们能够下载一个预训练好的英语语言模型，并且用它在自然语言处理的分类任务中取得顶尖的成果。（我们预计很快就会有更多语种的预训练模型可供使用——说不定等你读到这本书的时候，它们已经有了呢。）那么，我们为什么还要详细地学习如何训练一个语言模型呢？

当然，其中一个原因是，了解你正在使用的模型的基本原理是很有帮助的。但还有另一个非常实际的原因，那就是如果你在微调分类模型之前，先对（基于序列的）语言模型进行微调，效果会更好。比如说，在IMDb情感分析任务中，数据集里还包含另外5万条没有标注任何正面或负面情感的影评。因为训练集中有2.5万条带标注的影评，验证集中也有2.5万条，这样算下来总共有10万条影评。我们可以利用所有这些影评，对那个只在维基百科文章上训练过的预训练语言模型进行微调；这样就能得到一个特别擅长预测影评中下一个单词的语言模型。

这就是所谓的通用语言模型微调（ULMFit）方法。这篇[论文](https://arxiv.org/abs/1801.06146)指出，在迁移学习到分类任务之前，先对语言模型进行这一额外的微调步骤，能让预测结果大幅提升。采用这种方法的话，在自然语言处理的迁移学习过程中，我们就有三个阶段，具体总结见《ULMFit流程》。

<img alt="Diagram of the ULMFiT process" width="700" caption="The ULMFiT process" id="ulmfit_process" src="https://github.com/georgez9/fastbook/blob/master/images/att_00027.png?raw=1">

We'll now explore how to apply a neural network to this language modeling problem, using the concepts introduced in the last two chapters. But before reading further, pause and think about how *you* would approach this.

现在，我们将运用前两章中介绍的概念，探究如何将神经网络应用于这个语言建模问题。但在继续往下读之前，请先停下来思考一下，你自己会如何处理这个问题呢。

## Text Preprocessing

It's not at all obvious how we're going to use what we've learned so far to build a language model. Sentences can be different lengths, and documents can be very long. So, how can we predict the next word of a sentence using a neural network? Let's find out!

We've already seen how categorical variables can be used as independent variables for a neural network. The approach we took for a single categorical variable was to:

1. Make a list of all possible levels of that categorical variable (we'll call this list the *vocab*).
1. Replace each level with its index in the vocab.
1. Create an embedding matrix for this containing a row for each level (i.e., for each item of the vocab).
1. Use this embedding matrix as the first layer of a neural network. (A dedicated embedding matrix can take as inputs the raw vocab indexes created in step 2; this is equivalent to but faster and more efficient than a matrix that takes as input one-hot-encoded vectors representing the indexes.)

We can do nearly the same thing with text! What is new is the idea of a sequence. First we concatenate all of the documents in our dataset into one big long string and split it into words, giving us a very long list of words (or "tokens"). Our independent variable will be the sequence of words starting with the first word in our very long list and ending with the second to last, and our dependent variable will be the sequence of words starting with the second word and ending with the last word.

Our vocab will consist of a mix of common words that are already in the vocabulary of our pretrained model and new words specific to our corpus (cinematographic terms or actors names, for instance). Our embedding matrix will be built accordingly: for words that are in the vocabulary of our pretrained model, we will take the corresponding row in the embedding matrix of the pretrained model; but for new words we won't have anything, so we will just initialize the corresponding row with a random vector.

要如何运用我们目前所学的知识来构建一个语言模型，这一点一点也不明确。句子的长度可能各不相同，而且文档可能会非常长。那么，我们要怎样利用神经网络来预测一个句子的下一个单词呢？让我们一探究竟！

我们已经了解到分类变量是如何被用作神经网络的自变量的。对于单个分类变量，我们采取的方法是：
1. 列出该分类变量的所有可能取值（我们将这个列表称为 “词表”）。
1. 用该取值在词表中的索引来替换每个取值。
1. 为其创建一个嵌入矩阵，矩阵中为每个取值（即词表中的每个元素）都有一行。
1. 将这个嵌入矩阵用作神经网络的第一层。（专用的嵌入矩阵可以将在步骤2中创建的原始词表索引作为输入；这与将表示索引的独热编码向量作为输入的矩阵是等效的，但前者速度更快且效率更高。）

对于文本，我们几乎可以采取同样的做法！这里新出现的概念是序列。首先，我们将数据集中的所有文档连接成一个很长很长的字符串，然后将其拆分成单词，这样我们就得到了一个非常长的单词（或 “词元”）列表。我们的自变量将是从这个很长的列表的第一个单词开始，到倒数第二个单词结束的单词序列，而因变量将是从第二个单词开始，到最后一个单词结束的单词序列。

我们的词表将由两部分组成，一部分是已经存在于预训练模型词汇中的常用词，另一部分是我们语料库特有的新单词（比如电影相关术语或演员名字）。我们的嵌入矩阵也将相应地构建：对于存在于预训练模型词汇中的单词，我们将取用预训练模型嵌入矩阵中对应的行；但对于新单词，我们没有现成的内容，所以我们只需用一个随机向量来初始化嵌入矩阵中对应的行即可。

Each of the steps necessary to create a language model has jargon associated with it from the world of natural language processing, and fastai and PyTorch classes available to help. The steps are:

- Tokenization:: Convert the text into a list of words (or characters, or substrings, depending on the granularity of your model)
- Numericalization:: Make a list of all of the unique words that appear (the vocab), and convert each word into a number, by looking up its index in the vocab
- Language model data loader creation:: fastai provides an `LMDataLoader` class which automatically handles creating a dependent variable that is offset from the independent variable by one token. It also handles some important details, such as how to shuffle the training data in such a way that the dependent and independent variables maintain their structure as required
- Language model creation:: We need a special kind of model that does something we haven't seen before: handles input lists which could be arbitrarily big or small. There are a number of ways to do this; in this chapter we will be using a *recurrent neural network* (RNN). We will get to the details of these RNNs in the <<chapter_nlp_dive>>, but for now, you can think of it as just another deep neural network.

Let's take a look at how each step works in detail.

创建一个语言模型所需的每一个步骤，在自然语言处理领域都有与之相关的专业术语，并且有fastai和PyTorch的类可以提供帮助。这些步骤如下：

- **分词（Tokenization）**：将文本转换为单词（或者字符、子字符串，具体取决于模型的粒度）列表。
- **数值化（Numericalization）**：列出所有出现的唯一单词（即词表），并通过在词表中查找每个单词的索引，将其转换为一个数字。
- **创建语言模型数据加载器（Language model data loader creation）**：fastai提供了一个`LMDataLoader`类，它会自动处理创建一个因变量，该因变量与自变量之间相差一个词元。它还会处理一些重要的细节，例如如何以一种能让因变量和自变量按要求保持其结构的方式来打乱训练数据。
- **创建语言模型（Language model creation）**：我们需要一种特殊类型的模型，它能完成我们之前未曾见过的操作：处理大小任意的输入列表。有多种方法可以做到这一点；在本章中，我们将使用一种“循环神经网络”（RNN）。我们会在《深入自然语言处理章节》中详细探讨这些循环神经网络，但目前，你可以把它看作是另一种深度神经网络。

让我们详细了解一下每个步骤是如何工作的。

### Tokenization

When we said "convert the text into a list of words," we left out a lot of details. For instance, what do we do with punctuation? How do we deal with a word like "don't"? Is it one word, or two? What about long medical or chemical words? Should they be split into their separate pieces of meaning? How about hyphenated words? What about languages like German and Polish where we can create really long words from many, many pieces? What about languages like Japanese and Chinese that don't use bases at all, and don't really have a well-defined idea of *word*?

Because there is no one correct answer to these questions, there is no one approach to tokenization. There are three main approaches:

- Word-based:: Split a sentence on spaces, as well as applying language-specific rules to try to separate parts of meaning even when there are no spaces (such as turning "don't" into "do n't"). Generally, punctuation marks are also split into separate tokens.
- Subword based:: Split words into smaller parts, based on the most commonly occurring substrings. For instance, "occasion" might be tokenized as "o c ca sion."
- Character-based:: Split a sentence into its individual characters.

We'll be looking at word and subword tokenization here, and we'll leave character-based tokenization for you to implement in the questionnaire at the end of this chapter.

当我们说“将文本转换为单词列表”时，我们省略了很多细节。例如，我们该如何处理标点符号呢？像“don't”这样的单词，我们该怎么处理？它算一个单词还是两个单词呢？那些较长的医学或化学术语又该如何处理？是否应该把它们拆分成具有不同含义的各个部分呢？带有连字符的单词又该怎么处理呢？对于像德语和波兰语这样的语言，我们可以用许多部分组成非常长的单词，在这种情况下该如何处理呢？还有像日语和汉语这样根本不使用词基，而且实际上并没有一个明确界定的“单词”概念的语言，又该怎么办呢？

由于这些问题没有一个绝对正确的答案，所以也没有一种单一的分词方法。主要有三种分词方法：
- **基于单词的分词（Word-based）**：根据空格来拆分句子，同时应用特定语言的规则，即使没有空格也尝试将不同的语义部分分开（例如把“don't”拆分成“do n't”）。一般来说，标点符号也会被拆分成单独的词元。
- **基于子词的分词（Subword based）**：根据最常出现的子字符串，将单词拆分成更小的部分。例如，“occasion”可能会被分词为“o c ca sion”。
- **基于字符的分词（Character-based）**：将句子拆分成单个字符。

我们在这里将探讨基于单词和基于子词的分词方法，而基于字符的分词方法就留给你在本章末尾的问题部分去实践了。

> jargon: token: One element of a list created by the tokenization process. It could be a word, part of a word (a _subword_), or a single character.

> **术语**：词元（token）：在分词过程中所创建列表里的一个元素。它可以是一个单词、一个单词的一部分（即“子词”），或者是单个字符。

### Word Tokenization with fastai

Rather than providing its own tokenizers, fastai instead provides a consistent interface to a range of tokenizers in external libraries. Tokenization is an active field of research, and new and improved tokenizers are coming out all the time, so the defaults that fastai uses change too. However, the API and options shouldn't change too much, since fastai tries to maintain a consistent API even as the underlying technology changes.

Let's try it out with the IMDb dataset that we used in <<chapter_intro>>:

fastai 并没有自行开发分词器，而是为外部库中的一系列分词器提供了一个统一的接口。分词是一个活跃的研究领域，新的、更优的分词器不断涌现，所以 fastai 使用的默认分词器也会随之改变。不过，其应用程序编程接口（API）和选项不会有太大变化，因为即使底层技术发生改变，fastai 也会努力保持 API 的一致性。

让我们用在《章节简介》中使用过的互联网电影数据库（IMDb）数据集来试一试：

In [75]:
from fastai.text.all import *
path = untar_data(URLs.IMDB)

We'll need to grab the text files in order to try out a tokenizer. Just like `get_image_files`, which we've used many times already, gets all the image files in a path, `get_text_files` gets all the text files in a path. We can also optionally pass `folders` to restrict the search to a particular list of subfolders:

为了试用一个分词器，我们需要获取文本文件。就像我们已经多次使用过的 `get_image_files` 函数用于获取某个路径下的所有图像文件一样，`get_text_files` 函数用于获取某个路径下的所有文本文件。我们还可以选择性地传入 `folders` 参数，以便将搜索范围限制在特定的子文件夹列表内：

In [76]:
files = get_text_files(path, folders = ['train', 'test', 'unsup'])

Here's a review that we'll tokenize (we'll just print the start of it here to save space):

下面是一条我们将要进行分词的影评（为节省篇幅，我们在此只打印出它的开头部分）：

In [77]:
txt = files[0].open().read(); txt[:75]

"this is the first Simbhu movie i've seen and wow!i just keep watching this "

As we write this book, the default English word tokenizer for fastai uses a library called *spaCy*. It has a sophisticated rules engine with special rules for URLs, individual special English words, and much more. Rather than directly using `SpacyTokenizer`, however, we'll use `WordTokenizer`, since that will always point to fastai's current default word tokenizer (which may not necessarily be spaCy, depending when you're reading this).

Let's try it out. We'll use fastai's `coll_repr(collection, n)` function to display the results. This displays the first *`n`* items of *`collection`*, along with the full size—it's what `L` uses by default. Note that fastai's tokenizers take a collection of documents to tokenize, so we have to wrap `txt` in a list:

在我们撰写本书时，fastai 默认的英语单词分词器使用了一个名为 *spaCy* 的库。它有一个复杂的规则引擎，针对 URL、特殊的英语单词等都有专门的规则。不过，我们不会直接使用 `SpacyTokenizer`，而是会使用 `WordTokenizer`，因为它始终指向 fastai 当前默认的单词分词器（根据你阅读本书的时间，这个分词器不一定是 spaCy）。

让我们来试试看。我们将使用 fastai 的 `coll_repr(collection, n)` 函数来展示结果。该函数会显示集合 *`collection`* 的前 *`n`* 个元素，同时显示集合的完整大小，这也是 `L` 默认的显示方式。请注意，fastai 的分词器需要接收一个文档集合来进行分词，所以我们必须把 `txt` 放在一个列表中：

In [78]:
first?

In [79]:
coll_repr?

In [80]:
spacy = WordTokenizer()
toks = first(spacy([txt]))
print(coll_repr(toks, 30))

(#124) ['this','is','the','first','Simbhu','movie','i',"'ve",'seen','and','wow!i','just','keep','watching','this','movie','over','and','over','again.is','it','because','of','Simbhu','himself?is','it','the','songs?is','it','cuz'...]


As you see, spaCy has mainly just separated out the words and punctuation. But it does something else here too: it has split "it's" into "it" and "'s". That makes intuitive sense; these are separate words, really. Tokenization is a surprisingly subtle task, when you think about all the little details that have to be handled. Fortunately, spaCy handles these pretty well for us—for instance, here we see that "." is separated when it terminates a sentence, but not in an acronym or number:

如你所见，spaCy 主要就是把单词和标点符号分隔开了。但它在这里还做了另外一件事：它把 “it's” 拆分成了 “it” 和 “'s”。这很符合直觉，实际上它们确实是两个独立的单词。当你仔细思考分词时需要处理的所有小细节，就会发现这是一项相当微妙的任务。幸运的是，spaCy 帮我们处理得相当不错 —— 例如，在这里我们可以看到，当 “.” 用于句末时会被分隔开，但在缩写词或数字里则不会：

In [81]:
first(spacy(['The U.S. dollar $1 is $1.00.']))

(#9) ['The','U.S.','dollar','$','1','is','$','1.00','.']

fastai then adds some additional functionality to the tokenization process with the `Tokenizer` class:

然后，fastai 通过 `Tokenizer` 类为分词过程增添了一些额外的功能：

In [82]:
tkn = Tokenizer(spacy)
print(coll_repr(tkn(txt), 31))

(#133) ['xxbos','this','is','the','first','xxmaj','simbhu','movie','i',"'ve",'seen','and','wow!i','just','keep','watching','this','movie','over','and','over','again.is','it','because','of','xxmaj','simbhu','himself?is','it','the','songs?is'...]


Notice that there are now some tokens that start with the characters "xx", which is not a common word prefix in English. These are *special tokens*.

For example, the first item in the list, `xxbos`, is a special token that indicates the start of a new text ("BOS" is a standard NLP acronym that means "beginning of stream"). By recognizing this start token, the model will be able to learn it needs to "forget" what was said previously and focus on upcoming words.

These special tokens don't come from spaCy directly. They are there because fastai adds them by default, by applying a number of rules when processing text. These rules are designed to make it easier for a model to recognize the important parts of a sentence. In a sense, we are translating the original English language sequence into a simplified tokenized language—a language that is designed to be easy for a model to learn.

For instance, the rules will replace a sequence of four exclamation points with a special *repeated character* token, followed by the number four, and then a single exclamation point. In this way, the model's embedding matrix can encode information about general concepts such as repeated punctuation rather than requiring a separate token for every number of repetitions of every punctuation mark. Similarly, a capitalized word will be replaced with a special capitalization token, followed by the lowercase version of the word. This way, the embedding matrix only needs the lowercase versions of the words, saving compute and memory resources, but can still learn the concept of capitalization.

Here are some of the main special tokens you'll see:

- `xxbos`:: Indicates the beginning of a text (here, a review)
- `xxmaj`:: Indicates the next word begins with a capital (since we lowercased everything)
- `xxunk`:: Indicates the word is unknown

To see the rules that were used, you can check the default rules:

注意，现在列表中有一些以 “xx” 开头的词元，而 “xx” 在英语里并非常见的单词前缀。这些就是 “特殊词元”。

例如，列表中的第一个元素 `xxbos` 就是一个特殊词元，它表示一段新文本的开始（“BOS” 是自然语言处理领域的一个标准缩写，意思是 “流的开始”）。通过识别这个起始词元，模型就能知道需要 “忘掉” 之前的内容，转而关注接下来的单词。

这些特殊词元并非直接来自 spaCy。它们之所以存在，是因为 fastai 在处理文本时默认应用了一系列规则并添加了这些词元。这些规则旨在让模型更轻松地识别句子中的重要部分。从某种意义上说，我们是把原始的英语语言序列转化成了一种简化的分词语言，这种语言是专门为方便模型学习而设计的。

例如，规则会把连续四个感叹号替换成一个特殊的 “重复字符” 词元，后面跟着数字 4，然后是一个单独的感叹号。这样一来，模型的嵌入矩阵就能对诸如重复标点这类通用概念进行编码，而无需为每个标点符号的每一种重复次数都设置一个单独的词元。同样，一个首字母大写的单词会被替换成一个特殊的大写词元，后面跟着该单词的小写形式。通过这种方式，嵌入矩阵只需要存储单词的小写形式，从而节省计算和内存资源，但仍然能够学习到大写的概念。

以下是你会见到的一些主要特殊词元：
- `xxbos`：表示一段文本（这里指一篇影评）的开始。
- `xxmaj`：表示下一个单词以大写字母开头（因为我们把所有内容都转换成了小写）。
- `xxunk`：表示这个单词是未知的。

若想查看所使用的规则，你可以查看默认规则：

In [83]:
defaults.text_proc_rules

[<function fastai.text.core.fix_html(x)>,
 <function fastai.text.core.replace_rep(t)>,
 <function fastai.text.core.replace_wrep(t)>,
 <function fastai.text.core.spec_add_spaces(t)>,
 <function fastai.text.core.rm_useless_spaces(t)>,
 <function fastai.text.core.replace_all_caps(t)>,
 <function fastai.text.core.replace_maj(t)>,
 <function fastai.text.core.lowercase(t, add_bos=True, add_eos=False)>]

As always, you can look at the source code of each of them in a notebook by typing:

```
??replace_rep
```

Here is a brief summary of what each does:

- `fix_html`:: Replaces special HTML characters with a readable version (IMDb reviews have quite a few of these)
- `replace_rep`:: Replaces any character repeated three times or more with a special token for repetition (`xxrep`), the number of times it's repeated, then the character
- `replace_wrep`:: Replaces any word repeated three times or more with a special token for word repetition (`xxwrep`), the number of times it's repeated, then the word
- `spec_add_spaces`:: Adds spaces around / and #
- `rm_useless_spaces`:: Removes all repetitions of the space character
- `replace_all_caps`:: Lowercases a word written in all caps and adds a special token for all caps (`xxup`) in front of it
- `replace_maj`:: Lowercases a capitalized word and adds a special token for capitalized (`xxmaj`) in front of it
- `lowercase`:: Lowercases all text and adds a special token at the beginning (`xxbos`) and/or the end (`xxeos`)

和往常一样，你可以在笔记本中输入以下代码来查看每个函数的源代码：

```
??replace_rep
```

以下是每个规则作用的简要总结：

- `fix_html`：将特殊的 HTML 字符替换为易读的版本（IMDb 影评中有不少这样的字符）。
- `replace_rep`：将任何重复三次或更多次的字符替换为一个表示重复的特殊词元（`xxrep`）、重复的次数，然后是该字符。
- `replace_wrep`：将任何重复三次或更多次的单词替换为一个表示单词重复的特殊词元（`xxwrep`）、重复的次数，然后是该单词。
- `spec_add_spaces`：在 `/` 和 `#` 周围添加空格。
- `rm_useless_spaces`：去除所有重复的空格字符。
- `replace_all_caps`：将全大写的单词转换为小写，并在其前面添加一个表示全大写的特殊词元（`xxup`）。
- `replace_maj`：将首字母大写的单词转换为小写，并在其前面添加一个表示首字母大写的特殊词元（`xxmaj`）。
- `lowercase`：将所有文本转换为小写，并在开头添加一个特殊词元（`xxbos`）和/或在结尾添加一个特殊词元（`xxeos`）。

Let's take a look at a few of them in action:

让我们来看看其中一些（方法或工具）实际运行的情况吧。

In [84]:
coll_repr(tkn('&copy;   Fast.ai www.fast.ai/INDEX'), 31)

"(#11) ['xxbos','©','xxmaj','fast.ai','xxrep','3','w','.fast.ai','/','xxup','index']"

Now let's take a look at how subword tokenization would work.

现在，让我们来看看基于子词的分词是如何运作的吧。

### Subword Tokenization

In addition to the *word tokenization* approach seen in the last section, another popular tokenization method is *subword tokenization*. Word tokenization relies on an assumption that spaces provide a useful separation of components of meaning in a sentence. However, this assumption is not always appropriate. For instance, consider this sentence: 我的名字是郝杰瑞 ("My name is Jeremy Howard" in Chinese). That's not going to work very well with a word tokenizer, because there are no spaces in it! Languages like Chinese and Japanese don't use spaces, and in fact they don't even have a well-defined concept of a "word." There are also languages, like Turkish and Hungarian, that can add many subwords together without spaces, creating very long words that include a lot of separate pieces of information.

To handle these cases, it's generally best to use subword tokenization. This proceeds in two steps:

1. Analyze a corpus of documents to find the most commonly occurring groups of letters. These become the vocab.
2. Tokenize the corpus using this vocab of *subword units*.

Let's look at an example. For our corpus, we'll use the first 2,000 movie reviews:

除了上一节中介绍的“单词分词”方法外，另一种流行的分词方法是“子词分词”。单词分词基于这样一个假设，即空格能有效地将句子中的语义成分分隔开来。然而，这个假设并不总是合理的。例如，看这样一个句子：我的名字是郝杰瑞（中文意思是“我的名字是杰里米·霍华德”）。对于单词分词器来说，处理这个句子的效果不会太好，因为句子里没有空格！像中文和日文这样的语言不使用空格，实际上它们甚至没有一个明确界定的“单词”概念。还有一些语言，比如土耳其语和匈牙利语，它们可以在没有空格的情况下将许多子词组合在一起，形成包含大量独立信息的很长的单词。

为了处理这些情况，通常最好使用子词分词。这个过程分两步进行：
1. 分析一个文档语料库，找出最常出现的字母组合。这些组合就构成了词表。
2. 使用这个由“子词单元”组成的词表对语料库进行分词。

让我们来看一个例子。对于我们的语料库，我们将使用前 2000 条影评：

In [85]:
txts = L(o.open().read() for o in files[:2000])

We instantiate our tokenizer, passing in the size of the vocab we want to create, and then we need to "train" it. That is, we need to have it read our documents and find the common sequences of characters to create the vocab. This is done with `setup`. As we'll see shortly, `setup` is a special fastai method that is called automatically in our usual data processing pipelines. Since we're doing everything manually at the moment, however, we have to call it ourselves. Here's a function that does these steps for a given vocab size, and shows an example output:

我们实例化分词器，同时传入想要创建的词表的大小，然后需要对其进行“训练”。也就是说，我们要让它读取文档，找出常见的字符序列来创建词表。这可以通过 `setup` 方法来完成。很快我们会看到，`setup` 是 fastai 中的一个特殊方法，在常规的数据处理流程中它会自动被调用。不过，由于目前我们是手动完成所有操作，所以得自己调用这个方法。下面是一个函数，它会针对给定的词表大小完成上述步骤，并展示一个示例输出：

In [86]:
def subword(sz):
    sp = SubwordTokenizer(vocab_sz=sz)
    sp.setup(txts)
    return ' '.join(first(sp([txt]))[:40])

Let's try it out:

让我们试试看：

In [87]:
subword(1000)

"▁this ▁is ▁the ▁first ▁S im b h u ▁movie ▁i ' ve ▁seen ▁and ▁w ow ! i ▁just ▁keep ▁watching ▁this ▁movie ▁over ▁and ▁over ▁again . is ▁it ▁because ▁of ▁S im b h u ▁himself ?"

When using fastai's subword tokenizer, the special character `▁` represents a space character in the original text.

If we use a smaller vocab, then each token will represent fewer characters, and it will take more tokens to represent a sentence:

当使用 fastai 的子词分词器时，特殊字符 `▁` 表示原始文本中的空格字符。

如果我们使用较小的词表，那么每个词元将代表更少的字符，并且需要更多的词元来表示一个句子：

In [88]:
subword(200)

"▁this ▁is ▁the ▁f ir s t ▁S i m b h u ▁movie ▁ i ' ve ▁see n ▁and ▁w o w ! i ▁ j us t ▁ k e e p ▁w a t ch ing"

On the other hand, if we use a larger vocab, then most common English words will end up in the vocab themselves, and we will not need as many to represent a sentence:

另一方面，如果我们使用更大的词表，那么大多数常见的英语单词本身就会包含在词表中，这样我们就不需要那么多词元来表示一个句子了。

In [89]:
subword(10000)

"▁this ▁is ▁the ▁first ▁Sim bhu ▁movie ▁i ' ve ▁seen ▁and ▁wow ! i ▁just ▁keep ▁watching ▁this ▁movie ▁over ▁and ▁over ▁again . is ▁it ▁because ▁of ▁Sim bhu ▁himself ? is ▁it ▁the ▁songs ? is ▁it"

Picking a subword vocab size represents a compromise: a larger vocab means fewer tokens per sentence, which means faster training, less memory, and less state for the model to remember; but on the downside, it means larger embedding matrices, which require more data to learn.

Overall, subword tokenization provides a way to easily scale between character tokenization (i.e., using a small subword vocab) and word tokenization (i.e., using a large subword vocab), and handles every human language without needing language-specific algorithms to be developed. It can even handle other "languages" such as genomic sequences or MIDI music notation! For this reason, in the last year its popularity has soared, and it seems likely to become the most common tokenization approach (it may well already be, by the time you read this!).

选择子词词表的大小是一种权衡：更大的词表意味着每个句子的词元更少，这也就意味着训练速度更快、占用内存更少，并且模型需要记住的状态信息也更少；但不利的一面是，这意味着嵌入矩阵会更大，而更大的嵌入矩阵需要更多的数据来进行学习。

总的来说，子词分词提供了一种在字符分词（即使用较小的子词词表）和单词分词（即使用较大的子词词表）之间轻松扩展的方法，并且无需开发特定语言的算法就能处理各种人类语言。它甚至还能处理其他“语言”，比如基因组序列或 MIDI 音乐符号！正因如此，在过去的一年里，子词分词的受欢迎程度大幅上升，它很有可能会成为最常用的分词方法（当你读到这部分内容时，它很可能已经是了！）。

Once our texts have been split into tokens, we need to convert them to numbers. We'll look at that next.

一旦我们的文本被分割成了词元，我们就需要将它们转换为数字。接下来我们就来探讨这个问题。

### Numericalization with fastai

*Numericalization* is the process of mapping tokens to integers. The steps are basically identical to those necessary to create a `Category` variable, such as the dependent variable of digits in MNIST:

1. Make a list of all possible levels of that categorical variable (the vocab).
1. Replace each level with its index in the vocab.

Let's take a look at this in action on the word-tokenized text we saw earlier:

“数值化” 是将词元映射为整数的过程。其步骤与创建一个 `Category` 变量（比如 MNIST 数据集中数字对应的因变量）所需的步骤基本相同：

1. 列出该分类变量所有可能的取值（即词表）。
2. 用每个取值在词表中的索引来替换该取值。

下面让我们对之前见过的经过单词分词处理的文本进行实际操作，看看这个过程是怎样的：

In [90]:
toks = tkn(txt)
print(coll_repr(tkn(txt), 31))

(#133) ['xxbos','this','is','the','first','xxmaj','simbhu','movie','i',"'ve",'seen','and','wow!i','just','keep','watching','this','movie','over','and','over','again.is','it','because','of','xxmaj','simbhu','himself?is','it','the','songs?is'...]


Just like with `SubwordTokenizer`, we need to call `setup` on `Numericalize`; this is how we create the vocab. That means we'll need our tokenized corpus first. Since tokenization takes a while, it's done in parallel by fastai; but for this manual walkthrough, we'll use a small subset:

就像使用 `SubwordTokenizer` 一样，我们需要对 `Numericalize` 调用 `setup` 方法；通过这种方式来创建词表。这意味着我们首先需要有经过分词处理的语料库。由于分词过程需要一些时间，fastai 会并行处理分词操作；但在这个手动演示过程中，我们将使用一个较小的子集：

In [91]:
toks200 = txts[:200].map(tkn)
toks200[0]

(#133) ['xxbos','this','is','the','first','xxmaj','simbhu','movie','i',"'ve",'seen','and','wow!i','just','keep','watching','this','movie','over','and'...]

We can pass this to `setup` to create our vocab:

我们可以将这个（经过分词处理后的语料库）传递给 `setup` 方法，从而创建我们的词表。

In [92]:
num = Numericalize()
num.setup(toks200)
coll_repr(num.vocab,20)

"(#2136) ['xxunk','xxpad','xxbos','xxeos','xxfld','xxrep','xxwrep','xxup','xxmaj','the',',','.','a','and','of','to','is','it','in','i'...]"

Our special rules tokens appear first, and then every word appears once, in frequency order. The defaults to `Numericalize` are `min_freq=3,max_vocab=60000`. `max_vocab=60000` results in fastai replacing all words other than the most common 60,000 with a special *unknown word* token, `xxunk`. This is useful to avoid having an overly large embedding matrix, since that can slow down training and use up too much memory, and can also mean that there isn't enough data to train useful representations for rare words. However, this last issue is better handled by setting `min_freq`; the default `min_freq=3` means that any word appearing less than three times is replaced with `xxunk`.

fastai can also numericalize your dataset using a vocab that you provide, by passing a list of words as the `vocab` parameter.

Once we've created our `Numericalize` object, we can use it as if it were a function:

我们的特殊规则词元会首先出现，然后每个单词按出现频率顺序仅出现一次。`Numericalize` 的默认参数是 `min_freq = 3` 和 `max_vocab = 60000`。`max_vocab = 60000` 会让 fastai 把除最常见的 60000 个单词之外的所有单词替换成一个特殊的“未知单词”词元 `xxunk`。这有助于避免嵌入矩阵过大，因为过大的嵌入矩阵会减慢训练速度、占用过多内存，还可能导致没有足够的数据来为稀有单词训练出有用的表征。不过，最后这个问题通过设置 `min_freq` 能处理得更好；默认的 `min_freq = 3` 意味着任何出现次数少于三次的单词都会被替换成 `xxunk`。

fastai 还可以通过将一个单词列表作为 `vocab` 参数传入，使用你提供的词表对数据集进行数值化。

一旦我们创建了 `Numericalize` 对象，就可以像使用函数一样使用它：

In [93]:
nums = num(toks)[:20]; nums

TensorText([   2,   22,   16,    9,   86,    8, 1003,   27,   19,  177,  137,   13,    0,   56,  510,  157,   22,   27,  178,   13])

This time, our tokens have been converted to a tensor of integers that our model can receive. We can check that they map back to the original text:

这次，我们的词元已经被转换成了一个整数张量，我们的模型可以接收这个张量。我们可以检查一下，看看它们是否能映射回原始文本：

In [94]:
' '.join(num.vocab[o] for o in nums)

"xxbos this is the first xxmaj simbhu movie i 've seen and xxunk just keep watching this movie over and"

Now that we have numbers, we need to put them in batches for our model.

既然我们已经将文本转换为了数字，接下来就需要把这些数字整理成批次，以便我们的模型能够进行处理。

### Putting Our Texts into Batches for a Language Model

When dealing with images, we needed to resize them all to the same height and width before grouping them together in a mini-batch so they could stack together efficiently in a single tensor. Here it's going to be a little different, because one cannot simply resize text to a desired length. Also, we want our language model to read text in order, so that it can efficiently predict what the next word is. This means that each new batch should begin precisely where the previous one left off.

Suppose we have the following text:

> : In this chapter, we will go back over the example of classifying movie reviews we studied in chapter 1 and dig deeper under the surface. First we will look at the processing steps necessary to convert text into numbers and how to customize it. By doing this, we'll have another example of the PreProcessor used in the data block API.\nThen we will study how we build a language model and train it for a while.

The tokenization process will add special tokens and deal with punctuation to return this text:

> : xxbos xxmaj in this chapter , we will go back over the example of classifying movie reviews we studied in chapter 1 and dig deeper under the surface . xxmaj first we will look at the processing steps necessary to convert text into numbers and how to customize it . xxmaj by doing this , we 'll have another example of the preprocessor used in the data block xxup api . \n xxmaj then we will study how we build a language model and train it for a while .

We now have 90 tokens, separated by spaces. Let's say we want a batch size of 6. We need to break this text into 6 contiguous parts of length 15:

在处理图像时，我们需要先将所有图像调整为相同的高度和宽度，然后再将它们组合成一个小批次，这样才能有效地将它们堆叠在一个张量中。而在这里情况会有些不同，因为我们不能简单地将文本调整为所需的长度。此外，我们希望语言模型能够按顺序读取文本，以便它能有效地预测下一个单词是什么。这意味着每个新的批次都应该精确地从上一个批次结束的地方开始。

假设我们有以下文本：

> : In this chapter, we will go back over the example of classifying movie reviews we studied in chapter 1 and dig deeper under the surface. First we will look at the processing steps necessary to convert text into numbers and how to customize it. By doing this, we'll have another example of the PreProcessor used in the data block API.\nThen we will study how we build a language model and train it for a while.

分词过程会添加特殊词元并处理标点符号，从而得到以下文本：

> : xxbos xxmaj in this chapter , we will go back over the example of classifying movie reviews we studied in chapter 1 and dig deeper under the surface . xxmaj first we will look at the processing steps necessary to convert text into numbers and how to customize it . xxmaj by doing this , we 'll have another example of the preprocessor used in the data block xxup api . \n xxmaj then we will study how we build a language model and train it for a while .

现在我们有 90 个由空格分隔的词元。假设我们想要的批次大小为 6。我们需要将这段文本分成 6 个连续的部分，每个部分长度为 15：

In [95]:
#hide_input
stream = "In this chapter, we will go back over the example of classifying movie reviews we studied in chapter 1 and dig deeper under the surface. First we will look at the processing steps necessary to convert text into numbers and how to customize it. By doing this, we'll have another example of the PreProcessor used in the data block API.\nThen we will study how we build a language model and train it for a while."
tokens = tkn(stream)
bs,seq_len = 6,15
d_tokens = np.array([tokens[i*seq_len:(i+1)*seq_len] for i in range(bs)])
df = pd.DataFrame(d_tokens)
display(HTML(df.to_html(index=False,header=None)))

xxbos,xxmaj,in,this,chapter,",",we,will,go,back,over,the,example,of,classifying
movie,reviews,we,studied,in,chapter,1,and,dig,deeper,under,the,surface,.,xxmaj
first,we,will,look,at,the,processing,steps,necessary,to,convert,text,into,numbers,and
how,to,customize,it,.,xxmaj,by,doing,this,",",we,'ll,have,another,example
of,the,preprocessor,used,in,the,data,block,xxup,api,.,\n,xxmaj,then,we
will,study,how,we,build,a,language,model,and,train,it,for,a,while,.


In a perfect world, we could then give this one batch to our model. But that approach doesn't scale, because outside of this toy example it's unlikely that a single batch containing all the texts would fit in our GPU memory (here we have 90 tokens, but all the IMDb reviews together give several million).

So, we need to divide this array more finely into subarrays of a fixed sequence length. It is important to maintain order within and across these subarrays, because we will use a model that maintains a state so that it remembers what it read previously when predicting what comes next.

Going back to our previous example with 6 batches of length 15, if we chose a sequence length of 5, that would mean we first feed the following array:

在理想情况下，我们可以将这一个批次的数据提供给模型。但这种方法不具备扩展性，因为在实际应用中，除了这个简单的示例，包含所有文本的单个批次不太可能全部装入 GPU 内存（在这个例子中我们只有 90 个词元，但所有 IMDb 影评加起来有数百万个词元）。

因此，我们需要将这个数组更精细地划分为固定序列长度的子数组。在这些子数组内部和之间保持顺序非常重要，因为我们将使用一个能够维护状态的模型，这样它在预测后续内容时就能记住之前读取的信息。

回到之前那个分为 6 个长度为 15 的批次的例子，如果我们选择序列长度为 5，这意味着我们首先输入以下数组：

In [96]:
#hide_input
bs,seq_len = 6,5
d_tokens = np.array([tokens[i*15:i*15+seq_len] for i in range(bs)])
df = pd.DataFrame(d_tokens)
display(HTML(df.to_html(index=False,header=None)))

xxbos,xxmaj,in,this,chapter
movie,reviews,we,studied,in
first,we,will,look,at
how,to,customize,it,.
of,the,preprocessor,used,in
will,study,how,we,build


Then this one:

接下来是这个：

In [97]:
#hide_input
bs,seq_len = 6,5
d_tokens = np.array([tokens[i*15+seq_len:i*15+2*seq_len] for i in range(bs)])
df = pd.DataFrame(d_tokens)
display(HTML(df.to_html(index=False,header=None)))

",",we,will,go,back
chapter,1,and,dig,deeper
the,processing,steps,necessary,to
xxmaj,by,doing,this,","
the,data,block,xxup,api
a,language,model,and,train


And finally:

最后：

In [98]:
#hide_input
bs,seq_len = 6,5
d_tokens = np.array([tokens[i*15+10:i*15+15] for i in range(bs)])
df = pd.DataFrame(d_tokens)
display(HTML(df.to_html(index=False,header=None)))

over,the,example,of,classifying
under,the,surface,.,xxmaj
convert,text,into,numbers,and
we,'ll,have,another,example
.,\n,xxmaj,then,we
it,for,a,while,.


Going back to our movie reviews dataset, the first step is to transform the individual texts into a stream by concatenating them together. As with images, it's best to randomize the order of the inputs, so at the beginning of each epoch we will shuffle the entries to make a new stream (we shuffle the order of the documents, not the order of the words inside them, or the texts would not make sense anymore!).

We then cut this stream into a certain number of batches (which is our *batch size*). For instance, if the stream has 50,000 tokens and we set a batch size of 10, this will give us 10 mini-streams of 5,000 tokens. What is important is that we preserve the order of the tokens (so from 1 to 5,000 for the first mini-stream, then from 5,001 to 10,000...), because we want the model to read continuous rows of text (as in the preceding example). An `xxbos` token is added at the start of each during preprocessing, so that the model knows when it reads the stream when a new entry is beginning.

So to recap, at every epoch we shuffle our collection of documents and concatenate them into a stream of tokens. We then cut that stream into a batch of fixed-size consecutive mini-streams. Our model will then read the mini-streams in order, and thanks to an inner state, it will produce the same activation whatever sequence length we picked.

This is all done behind the scenes by the fastai library when we create an `LMDataLoader`. We do this by first applying our `Numericalize` object to the tokenized texts:

让我们再回到电影评论数据集，第一步是将每一篇文本连接起来，把它们转化为一个连续的词元流。和处理图像数据时一样，最好对输入数据的顺序进行随机化处理。所以在每个训练轮次开始时，我们会打乱数据项的顺序，从而形成一个新的词元流（注意，我们打乱的是文档的顺序，而不是文档内部单词的顺序，否则文本就会变得毫无意义！）。

接着，我们要把这个词元流分割成一定数量的批次，这个数量就是我们所说的“批次大小”。例如，如果词元流中有 50000 个词元，我们把批次大小设置为 10，那么就会得到 10 个包含 5000 个词元的子词元流。这里非常重要的一点是，我们要保持词元的顺序不变（也就是说，第一个子词元流包含从第 1 到第 5000 个词元，接着第二个子词元流包含从第 5001 到第 10000 个词元，依此类推），因为我们希望模型能够按顺序读取连续的文本行（就像前面例子中那样）。在预处理阶段，会在每篇文本的开头添加一个 `xxbos` 词元，这样模型在读取词元流时就能知道何时开始处理一篇新的文档。

总结一下，在每个训练轮次中，我们会打乱文档集合的顺序，然后把它们连接成一个词元流。接着，我们把这个词元流分割成一批固定大小的连续子词元流。模型会按顺序读取这些子词元流，并且由于模型具有内部状态，无论我们选择的序列长度是多少，它都能产生相同的激活结果。

当我们使用 fastai 库创建一个 `LMDataLoader` 时，上述所有操作都会在后台自动完成。我们通过将 `Numericalize` 对象应用于经过分词处理的文本来实现这一点：

In [99]:
nums200 = toks200.map(num)

and then passing that to `LMDataLoader`:

然后将其传递给 LMDataLoader：

In [100]:
dl = LMDataLoader(nums200)

Let's confirm that this gives the expected results, by grabbing the first batch:

让我们通过获取第一个批次的数据来确认这是否能得到预期的结果。

In [101]:
x,y = first(dl)
x.shape,y.shape

(torch.Size([64, 72]), torch.Size([64, 72]))

and then looking at the first row of the independent variable, which should be the start of the first text:

然后查看自变量的第一行，它应该是第一篇文本的起始部分。

In [102]:
' '.join(num.vocab[o] for o in x[0][:20])

"xxbos this is the first xxmaj simbhu movie i 've seen and xxunk just keep watching this movie over and"

The dependent variable is the same thing offset by one token:

因变量是将自变量偏移一个词元后得到的结果。

In [103]:
' '.join(num.vocab[o] for o in y[0][:20])

"this is the first xxmaj simbhu movie i 've seen and xxunk just keep watching this movie over and over"

This concludes all the preprocessing steps we need to apply to our data. We are now ready to train our text classifier.

至此，我们需要对数据应用的所有预处理步骤都已完成。现在我们已经准备好训练我们的文本分类器了。

## Training a Text Classifier

As we saw at the beginning of this chapter, there are two steps to training a state-of-the-art text classifier using transfer learning: first we need to fine-tune our language model pretrained on Wikipedia to the corpus of IMDb reviews, and then we can use that model to train a classifier.

As usual, let's start with assembling our data.

正如我们在本章开头所看到的，使用迁移学习来训练一个先进的文本分类器需要两个步骤：首先，我们需要将在维基百科上预训练的语言模型在IMDb影评数据集上进行微调，然后我们可以使用微调后的模型来训练一个分类器。

和往常一样，让我们从整理数据开始。

### Language Model Using DataBlock

fastai handles tokenization and numericalization automatically when `TextBlock` is passed to `DataBlock`. All of the arguments that can be passed to `Tokenize` and `Numericalize` can also be passed to `TextBlock`. In the next chapter we'll discuss the easiest ways to run each of these steps separately, to ease debugging—but you can always just debug by running them manually on a subset of your data as shown in the previous sections. And don't forget about `DataBlock`'s handy `summary` method, which is very useful for debugging data issues.

Here's how we use `TextBlock` to create a language model, using fastai's defaults:

当把 `TextBlock` 传递给 `DataBlock` 时，fastai 会自动处理分词和数值化。所有可以传递给 `Tokenize` 和 `Numericalize` 的参数，也都能传递给 `TextBlock`。在下一章，我们会讨论分别运行这些步骤的最简单方法，以方便调试。不过，你始终可以像前面章节所展示的那样，在数据子集上手动运行这些步骤来进行调试。另外，别忘了 `DataBlock` 实用的 `summary` 方法，它对于调试数据问题非常有用。

以下是我们如何使用 `TextBlock` 并借助 fastai 的默认设置来创建一个语言模型的示例：

In [104]:
get_imdb = partial(get_text_files, folders=['train', 'test', 'unsup'])

dls_lm = DataBlock(
    blocks=TextBlock.from_folder(path, is_lm=True),
    get_items=get_imdb, splitter=RandomSplitter(0.1)
).dataloaders(path, path=path, bs=128, seq_len=80)

FileNotFoundError: [Errno 2] No such file or directory: '/root/.fastai/data/imdb_tok/counter.pkl'

One thing that's different to previous types we've used in `DataBlock` is that we're not just using the class directly (i.e., `TextBlock(...)`, but instead are calling a *class method*. A class method is a Python method that, as the name suggests, belongs to a *class* rather than an *object*. (Be sure to search online for more information about class methods if you're not familiar with them, since they're commonly used in many Python libraries and applications; we've used them a few times previously in the book, but haven't called attention to them.) The reason that `TextBlock` is special is that setting up the numericalizer's vocab can take a long time (we have to read and tokenize every document to get the vocab). To be as efficient as possible it performs a few optimizations:

- It saves the tokenized documents in a temporary folder, so it doesn't have to tokenize them more than once
- It runs multiple tokenization processes in parallel, to take advantage of your computer's CPUs

We need to tell `TextBlock` how to access the texts, so that it can do this initial preprocessing—that's what `from_folder` does.

`show_batch` then works in the usual way:

与我们之前在 `DataBlock` 中使用的其他类型不同的是，我们并非直接使用类（即 `TextBlock(...)`），而是调用了一个*类方法*。类方法是 Python 中的一种方法，顾名思义，它属于*类*而非*对象*。（如果你对类方法不太熟悉，一定要上网搜索更多相关信息，因为它们在许多 Python 库和应用程序中很常用；在本书前面的内容中我们也曾多次使用过类方法，但并未特别强调。）`TextBlock` 的特殊之处在于，设置数值转换器的词表可能会花费很长时间（我们必须读取并对每篇文档进行分词处理才能得到词表）。为了尽可能提高效率，它进行了一些优化：

- 它会将分词后的文档保存到一个临时文件夹中，这样就无需对它们进行多次分词处理。
- 它会并行运行多个分词进程，以充分利用你计算机的 CPU。

我们需要告知 `TextBlock` 如何访问文本，这样它才能进行初始的预处理工作，而 `from_folder` 方法就起到了这个作用。

之后，`show_batch` 就能像往常一样工作了：

In [ ]:
dls_lm.show_batch(max_n=2)

In [ ]:
dls_lm.show_batch?

Now that our data is ready, we can fine-tune the pretrained language model.

既然我们的数据已经准备就绪，我们就可以对预训练的语言模型进行微调了。

### Fine-Tuning the Language Model

To convert the integer word indices into activations that we can use for our neural network, we will use embeddings, just like we did for collaborative filtering and tabular modeling. Then we'll feed those embeddings into a *recurrent neural network* (RNN), using an architecture called *AWD-LSTM* (we will show you how to write such a model from scratch in <<chapter_nlp_dive>>). As we discussed earlier, the embeddings in the pretrained model are merged with random embeddings added for words that weren't in the pretraining vocabulary. This is handled automatically inside `language_model_learner`:

为了将整数形式的单词索引转换为可用于神经网络的激活值，我们会像在协同过滤和表格数据建模时那样，使用嵌入层。然后，我们会把这些嵌入向量输入到一个*循环神经网络*（RNN）中，采用的架构是*AWD - LSTM*（我们会在<<自然语言处理深入剖析>>这一章向你展示如何从头开始编写这样一个模型）。正如我们之前所讨论的，预训练模型中的嵌入向量会与为预训练词表中不存在的单词所添加的随机嵌入向量合并。这一过程在 `language_model_learner` 内部会自动处理：

In [ ]:
learn = language_model_learner(
    dls_lm, AWD_LSTM, drop_mult=0.3,
    metrics=[accuracy, Perplexity()]).to_fp16()

The loss function used by default is cross-entropy loss, since we essentially have a classification problem (the different categories being the words in our vocab). The *perplexity* metric used here is often used in NLP for language models: it is the exponential of the loss (i.e., `torch.exp(cross_entropy)`). We  also include the accuracy metric, to see how many times our model is right when trying to predict the next word, since cross-entropy (as we've seen) is both hard to interpret, and tells us more about the model's confidence than its accuracy.

Let's go back to the process diagram from the beginning of this chapter. The first arrow has been completed for us and made available as a pretrained model in fastai, and we've just built the `DataLoaders` and `Learner` for the second stage. Now we're ready to fine-tune our language model!

默认使用的损失函数是交叉熵损失，因为本质上我们面对的是一个分类问题（不同的类别就是我们词表中的单词）。这里使用的“困惑度”指标在自然语言处理的语言模型中经常用到：它是损失值的指数（即 `torch.exp(cross_entropy)` ）。我们还纳入了准确率指标，以便了解在尝试预测下一个单词时我们的模型有多少次是正确的，因为正如我们所看到的，交叉熵既难以解释，而且它更多地告诉我们的是模型的置信度，而非准确率。

让我们回到本章开头的处理流程图。对于我们来说，第一个箭头所代表的步骤已经完成，并且在 fastai 中作为预训练模型可供使用，而且我们刚刚为第二个阶段构建了 `DataLoaders`（数据加载器）和 `Learner`（学习器）。现在我们已经准备好对我们的语言模型进行微调了！

<img alt="Diagram of the ULMFiT process" width="450" src="https://github.com/georgez9/fastbook/blob/master/images/att_00027.png?raw=1">

It takes quite a while to train each epoch, so we'll be saving the intermediate model results during the training process. Since `fine_tune` doesn't do that for us, we'll use `fit_one_cycle`. Just like `vision_learner`, `language_model_learner` automatically calls `freeze` when using a pretrained model (which is the default), so this will only train the embeddings (the only part of the model that contains randomly initialized weights—i.e., embeddings for words that are in our IMDb vocab, but aren't in the pretrained model vocab):

每个训练轮次（epoch）都需要花费相当长的时间，所以我们会在训练过程中保存中间的模型结果。由于 `fine_tune` 方法不会为我们完成这一操作，因此我们将使用 `fit_one_cycle` 方法。就像 `vision_learner` 一样，`language_model_learner` 在使用预训练模型（这是默认设置）时会自动调用 `freeze` 方法。所以，这将只会训练嵌入层（嵌入层是模型中唯一包含随机初始化权重的部分，也就是针对那些存在于我们的IMDb词表中，但不在预训练模型词表里的单词所对应的嵌入层）：

In [ ]:
print(torch.cuda.memory_allocated())
print(torch.cuda.memory_reserved())

In [ ]:
learn.fit_one_cycle(1, 2e-2)

This model takes a while to train, so it's a good opportunity to talk about saving intermediary results.

这个模型需要一段时间来训练，所以这是一个谈论保存中间结果的好机会。

### Saving and Loading Models

You can easily save the state of your model like so:

In [ ]:
learn.save('1epoch')

This will create a file in `learn.path/models/` named *1epoch.pth*. If you want to load your model in another machine after creating your `Learner` the same way, or resume training later, you can load the content of this file with:

In [ ]:
learn = learn.load('1epoch')

Once the initial training has completed, we can continue fine-tuning the model after unfreezing:

In [ ]:
learn.unfreeze()
learn.fit_one_cycle(10, 2e-3)

Once this is done, we save all of our model except the final layer that converts activations to probabilities of picking each token in our vocabulary. The model not including the final layer is called the *encoder*. We can save it with `save_encoder`:

In [ ]:
learn.save_encoder('finetuned')

> jargon: Encoder: The model not including the task-specific final layer(s). This term means much the same thing as _body_ when applied to vision CNNs, but "encoder" tends to be more used for NLP and generative models.

This completes the second stage of the text classification process: fine-tuning the language model. We can now use it to fine-tune a classifier using the IMDb sentiment labels.

### Text Generation

Before we move on to fine-tuning the classifier, let's quickly try something different: using our model to generate random reviews. Since it's trained to guess what the next word of the sentence is, we can use the model to write new reviews:

In [ ]:
TEXT = "I liked this movie because"
N_WORDS = 40
N_SENTENCES = 2
preds = [learn.predict(TEXT, N_WORDS, temperature=0.75)
         for _ in range(N_SENTENCES)]

In [ ]:
print("\n".join(preds))

As you can see, we add some randomness (we pick a random word based on the probabilities returned by the model) so we don't get exactly the same review twice. Our model doesn't have any programmed knowledge of the structure of a sentence or grammar rules, yet it has clearly learned a lot about English sentences: we can see it capitalizes properly (*I* is just transformed to *i* because our rules require two characters or more to consider a word as capitalized, so it's normal to see it lowercased) and is using consistent tense. The general review makes sense at first glance, and it's only if you read carefully that you can notice something is a bit off. Not bad for a model trained in a couple of hours!

But our end goal wasn't to train a model to generate reviews, but to classify them... so let's use this model to do just that.

### Creating the Classifier DataLoaders

We're now moving from language model fine-tuning to classifier fine-tuning. To recap, a language model predicts the next word of a document, so it doesn't need any external labels. A classifier, however, predicts some external label—in the case of IMDb, it's the sentiment of a document.

This means that the structure of our `DataBlock` for NLP classification will look very familiar. It's actually nearly the same as we've seen for the many image classification datasets we've worked with:

In [ ]:
dls_clas = DataBlock(
    blocks=(TextBlock.from_folder(path, vocab=dls_lm.vocab),CategoryBlock),
    get_y = parent_label,
    get_items=partial(get_text_files, folders=['train', 'test']),
    splitter=GrandparentSplitter(valid_name='test')
).dataloaders(path, path=path, bs=128, seq_len=72)

Just like with image classification, `show_batch` shows the dependent variable (sentiment, in this case) with each independent variable (movie review text):

In [ ]:
dls_clas.show_batch(max_n=3)

Looking at the `DataBlock` definition, every piece is familiar from previous data blocks we've built, with two important exceptions:

- `TextBlock.from_folder` no longer has the `is_lm=True` parameter.
- We pass the `vocab` we created for the language model fine-tuning.

The reason that we pass the `vocab` of the language model is to make sure we use the same correspondence of token to index. Otherwise the embeddings we learned in our fine-tuned language model won't make any sense to this model, and the fine-tuning step won't be of any use.

By passing `is_lm=False` (or not passing `is_lm` at all, since it defaults to `False`) we tell `TextBlock` that we have regular labeled data, rather than using the next tokens as labels. There is one challenge we have to deal with, however, which is to do with collating multiple documents into a mini-batch. Let's see with an example, by trying to create a mini-batch containing the first 10 documents. First we'll numericalize them:

In [ ]:
nums_samp = toks200[:10].map(num)

Let's now look at how many tokens each of these 10 movie reviews have:

In [ ]:
nums_samp.map(len)

Remember, PyTorch `DataLoader`s need to collate all the items in a batch into a single tensor, and a single tensor has a fixed shape (i.e., it has some particular length on every axis, and all items must be consistent). This should sound familiar: we had the same issue with images. In that case, we used cropping, padding, and/or squishing to make all the inputs the same size. Cropping might not be a good idea for documents, because it seems likely we'd remove some key information (having said that, the same issue is true for images, and we use cropping there; data augmentation hasn't been well explored for NLP yet, so perhaps there are actually opportunities to use cropping in NLP too!). You can't really "squish" a document. So that leaves padding!

We will expand the shortest texts to make them all the same size. To do this, we use a special padding token that will be ignored by our model. Additionally, to avoid memory issues and improve performance, we will batch together texts that are roughly the same lengths (with some shuffling for the training set). We do this by (approximately, for the training set) sorting the documents by length prior to each epoch. The result of this is that the documents collated into a single batch will tend to be of similar lengths. We won't pad every batch to the same size, but will instead use the size of the largest document in each batch as the target size. (It is possible to do something similar with images, which is especially useful for irregularly sized rectangular images, but at the time of writing no library provides good support for this yet, and there aren't any papers covering it. It's something we're planning to add to fastai soon, however, so keep an eye on the book's website; we'll add information about this as soon as we have it working well.)

The sorting and padding are automatically done by the data block API for us when using a `TextBlock`, with `is_lm=False`. (We don't have this same issue for language model data, since we concatenate all the documents together first, and then split them into equally sized sections.)

We can now create a model to classify our texts:

In [ ]:
learn = text_classifier_learner(dls_clas, AWD_LSTM, drop_mult=0.5,
                                metrics=accuracy).to_fp16()

The final step prior to training the classifier is to load the encoder from our fine-tuned language model. We use `load_encoder` instead of `load` because we only have pretrained weights available for the encoder; `load` by default raises an exception if an incomplete model is loaded:

In [ ]:
learn = learn.load_encoder('finetuned')

### Fine-Tuning the Classifier

The last step is to train with discriminative learning rates and *gradual unfreezing*. In computer vision we often unfreeze the model all at once, but for NLP classifiers, we find that unfreezing a few layers at a time makes a real difference:

In [ ]:
learn.fit_one_cycle(1, 2e-2)

In just one epoch we get the same result as our training in <<chapter_intro>>: not too bad! We can pass `-2` to `freeze_to` to freeze all except the last two parameter groups:

In [ ]:
learn.freeze_to(-2)
learn.fit_one_cycle(1, slice(1e-2/(2.6**4),1e-2))

Then we can unfreeze a bit more, and continue training:

In [ ]:
learn.freeze_to(-3)
learn.fit_one_cycle(1, slice(5e-3/(2.6**4),5e-3))

And finally, the whole model!

In [ ]:
learn.unfreeze()
learn.fit_one_cycle(2, slice(1e-3/(2.6**4),1e-3))

We reached 94.3% accuracy, which was state-of-the-art performance just three years ago. By training another model on all the texts read backwards and averaging the predictions of those two models, we can even get to 95.1% accuracy, which was the state of the art introduced by the ULMFiT paper. It was only beaten a few months ago, by fine-tuning a much bigger model and using expensive data augmentation techniques (translating sentences in another language and back, using another model for translation).

Using a pretrained model let us build a fine-tuned language model that was pretty powerful, to either generate fake reviews or help classify them. This is exciting stuff, but it's good to remember that this technology can also be used for malign purposes.

## Disinformation and Language Models

Even simple algorithms based on rules, before the days of widely available deep learning language models, could be used to create fraudulent accounts and try to influence policymakers. Jeff Kao, now a computational journalist at ProPublica, analyzed the comments that were sent to the US Federal Communications Commission (FCC) regarding a 2017 proposal to repeal net neutrality. In his article ["More than a Million Pro-Repeal Net Neutrality Comments Were Likely Faked"](https://hackernoon.com/more-than-a-million-pro-repeal-net-neutrality-comments-were-likely-faked-e9f0e3ed36a6), he reports how he discovered a large cluster of comments opposing net neutrality that seemed to have been generated by some sort of Mad Libs-style mail merge. In <<disinformation>>, the fake comments have been helpfully color-coded by Kao to highlight their formulaic nature.

<img src="https://github.com/georgez9/fastbook/blob/master/images/ethics/image16.png?raw=1" width="700" id="disinformation" caption="Comments received by the FCC during the net neutrality debate">

Kao estimated that "less than 800,000 of the 22M+ comments… could be considered truly unique" and that "more than 99% of the truly unique comments were in favor of keeping net neutrality."

Given advances in language modeling that have occurred since 2017, such fraudulent campaigns could be nearly impossible to catch now.  You now have all the necessary tools at your disposal to create a compelling language model—that is, something that can generate context-appropriate, believable text. It won't necessarily be perfectly accurate or correct, but it will be plausible. Think about what this technology would mean when put together with the kinds of disinformation campaigns we have learned about in recent years. Take a look at the Reddit dialogue shown in <<ethics_reddit>>, where a language model based on OpenAI's GPT-2 algorithm is having a conversation with itself about whether the US government should cut defense spending.

<img src="https://github.com/georgez9/fastbook/blob/master/images/ethics/image14.png?raw=1" id="ethics_reddit" caption="An algorithm talking to itself on Reddit" alt="An algorithm talking to itself on Reddit" width="600">

In this case, it was explicitly said that an algorithm was used, but imagine what would happen if a bad actor decided to release such an algorithm across social networks. They could do it slowly and carefully, allowing the algorithm to gradually develop followers and trust over time. It would not take many resources to have literally millions of accounts doing this. In such a situation we could easily imagine getting to a point where the vast majority of discourse online was from bots, and nobody would have any idea that it was happening.

We are already starting to see examples of machine learning being used to generate identities. For example, <<katie_jones>> shows a LinkedIn profile for Katie Jones.

<img src="https://github.com/georgez9/fastbook/blob/master/images/ethics/image15.jpeg?raw=1" width="400" id="katie_jones" caption="Katie Jones's LinkedIn profile">

Katie Jones was connected on LinkedIn to several members of mainstream Washington think tanks. But she didn't exist. That image you see was auto-generated by a generative adversarial network, and somebody named Katie Jones has not, in fact, graduated from the Center for Strategic and International Studies.

Many people assume or hope that algorithms will come to our defense here—that we will develop classification algorithms that can automatically recognise autogenerated content. The problem, however, is that this will always be an arms race, in which better classification (or discriminator) algorithms can be used to create better generation algorithms.

## Conclusion

In this chapter we explored the last application covered out of the box by the fastai library: text. We saw two types of models: language models that can generate texts, and a classifier that determines if a review is positive or negative. To build a state-of-the art classifier, we used a pretrained language model, fine-tuned it to the corpus of our task, then used its body (the encoder) with a new head to do the classification.

Before we end this section, we'll take a look at how the fastai library can help you assemble your data for your specific problems.

## Questionnaire

1. What is "self-supervised learning"?
1. What is a "language model"?
1. Why is a language model considered self-supervised?
1. What are self-supervised models usually used for?
1. Why do we fine-tune language models?
1. What are the three steps to create a state-of-the-art text classifier?
1. How do the 50,000 unlabeled movie reviews help us create a better text classifier for the IMDb dataset?
1. What are the three steps to prepare your data for a language model?
1. What is "tokenization"? Why do we need it?
1. Name three different approaches to tokenization.
1. What is `xxbos`?
1. List four rules that fastai applies to text during tokenization.
1. Why are repeated characters replaced with a token showing the number of repetitions and the character that's repeated?
1. What is "numericalization"?
1. Why might there be words that are replaced with the "unknown word" token?
1. With a batch size of 64, the first row of the tensor representing the first batch contains the first 64 tokens for the dataset. What does the second row of that tensor contain? What does the first row of the second batch contain? (Careful—students often get this one wrong! Be sure to check your answer on the book's website.)
1. Why do we need padding for text classification? Why don't we need it for language modeling?
1. What does an embedding matrix for NLP contain? What is its shape?
1. What is "perplexity"?
1. Why do we have to pass the vocabulary of the language model to the classifier data block?
1. What is "gradual unfreezing"?
1. Why is text generation always likely to be ahead of automatic identification of machine-generated texts?

### Further Research

1. See what you can learn about language models and disinformation. What are the best language models today? Take a look at some of their outputs. Do you find them convincing? How could a bad actor best use such a model to create conflict and uncertainty?
1. Given the limitation that models are unlikely to be able to consistently recognize machine-generated texts, what other approaches may be needed to handle large-scale disinformation campaigns that leverage deep learning?